# Analysis 2 — Review Analysis
**Phase 3 | Quick Commerce Operations Project**

Business questions answered here:
- How do delayed deliveries impact reviews?
- Which categories have worst fulfillment?
- What themes appear in negative reviews?

Input: `workspace.default.final_quick_comm_dataset` as `df`  
Outputs: Tableau-ready CSVs in `/outputs/`

## 0. Imports & Setup

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import warnings, os
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f9f9f7',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'grid.linestyle':   '--',
    'font.size':        11,
})
PALETTE = ['#E8593C','#F2A623','#3B8BD4','#1D9E75','#7F77DD','#888780']
print('Setup complete.')

## 1. Data Preparation

In [0]:
# Assume df is already loaded as workspace.default.final_quick_comm_dataset
# If running standalone, uncomment:
# df = pd.read_csv('data/final_quick_comm_dataset.csv', parse_dates=[
#     'order_purchase_timestamp','order_delivered_customer_date',
#     'order_estimated_delivery_date','review_creation_date'])

df = spark.table(
    "workspace.default.final_quick_comm_dataset"
).toPandas()

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])


print(f'Total rows   : {len(df):,}')
print(f'Unique orders: {df["order_id"].nunique():,}')
print(f'Review score null%: {df["review_score"].isna().mean()*100:.1f}%')

# Dedup: one row per order (multi-item orders repeat review columns)
review_df = df.drop_duplicates(subset='order_id', keep='first').copy()

# Score bins
review_df['score_bin'] = pd.cut(
    review_df['review_score'],
    bins=[0, 2, 3, 5],
    labels=['Negative (1-2)', 'Neutral (3)', 'Positive (4-5)']
)

# Delay buckets
def delay_bucket(d):
    if pd.isna(d):  return 'Unknown'
    if d <= 0:      return 'On time / Early'
    if d <= 3:      return '1-3 days late'
    if d <= 7:      return '4-7 days late'
    if d <= 14:     return '8-14 days late'
    return '15+ days late'

DELAY_ORDER = ['On time / Early','1-3 days late','4-7 days late',
               '8-14 days late','15+ days late','Unknown']

review_df['delay_bucket'] = pd.Categorical(
    review_df['delivery_delay_days'].apply(delay_bucket),
    categories=DELAY_ORDER, ordered=True
)

# Year-month for time trends
review_df['year_month'] = review_df['order_purchase_timestamp'].dt.to_period('M')

# Combine text fields for NLP
review_df['full_review_text'] = (
    review_df['review_comment_title'].fillna('')
    + ' '
    + review_df['review_comment_message'].fillna('')
).str.strip()

print(f'Deduped rows : {len(review_df):,}')
print('Score distribution:')
print(review_df['review_score'].value_counts().sort_index())

## 2. Review Score Distribution

In [0]:
avg_score    = review_df['review_score'].mean()
pct_negative = (review_df['score_bin'] == 'Negative (1-2)').mean() * 100
pct_positive = (review_df['score_bin'] == 'Positive (4-5)').mean() * 100
has_text_pct = (review_df['full_review_text'].str.len() > 2).mean() * 100

print('── Review KPIs ──────────────────────────────')
print(f'  Avg review score  : {avg_score:.2f} / 5.0')
print(f'  % Negative (1-2*) : {pct_negative:.1f}%')
print(f'  % Positive (4-5*) : {pct_positive:.1f}%')
print(f'  % Reviews w/ text : {has_text_pct:.1f}%')

kpi_df = pd.DataFrame([{
    'avg_review_score': round(avg_score, 2),
    'pct_negative':     round(pct_negative, 1),
    'pct_positive':     round(pct_positive, 1),
    'pct_has_text':     round(has_text_pct, 1),
    'total_orders':     len(review_df),
}])
kpi_df.to_csv('outputs/review_kpis.csv', index=False)

score_counts = review_df['review_score'].value_counts().sort_index()
score_pct    = score_counts / score_counts.sum() * 100
bar_colors   = ['#E8593C','#F2A623','#888780','#1D9E75','#0F6E56']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(score_counts.index, score_pct.values,
              color=bar_colors, edgecolor='white', linewidth=0.8, width=0.6)
for bar, val in zip(bars, score_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Review score (stars)')
ax.set_ylabel('% of orders')
ax.set_title(f'Review score distribution  |  Avg = {avg_score:.2f} stars', pad=12)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xticks([1,2,3,4,5])
plt.tight_layout()
plt.savefig('outputs/fig_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Delivery Delay Impact on Reviews

In [0]:
delay_impact = (
    review_df
    .groupby('delay_bucket', observed=True)
    .agg(
        order_count    = ('order_id',            'count'),
        avg_score      = ('review_score',         'mean'),
        pct_negative   = ('score_bin',
                          lambda x: (x == 'Negative (1-2)').mean() * 100),
        avg_delay_days = ('delivery_delay_days',  'mean'),
    )
    .reset_index()
)
delay_impact.columns.name = None
delay_impact['avg_score']    = delay_impact['avg_score'].round(2)
delay_impact['pct_negative'] = delay_impact['pct_negative'].round(1)

print('Delay impact on reviews:')
print(delay_impact.to_string(index=False))
delay_impact.to_csv('outputs/delay_impact.csv', index=False)

plot_data = delay_impact[delay_impact['delay_bucket'] != 'Unknown'].copy()
fig, ax1  = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
x = range(len(plot_data))

ax1.bar(x, plot_data['avg_score'], color='#3B8BD4', alpha=0.75,
        label='Avg review score', width=0.4)
ax2.plot(x, plot_data['pct_negative'], color='#E8593C', marker='o',
         linewidth=2, markersize=7, label='% Negative reviews')

ax1.set_xticks(list(x))
ax1.set_xticklabels(plot_data['delay_bucket'], rotation=15, ha='right')
ax1.set_ylabel('Avg review score', color='#3B8BD4')
ax1.set_ylim(0, 5.5)
ax2.set_ylabel('% Negative reviews (1-2 stars)', color='#E8593C')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())

l1, lb1 = ax1.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, lb1 + lb2, loc='upper right')
ax1.set_title('How delivery delay impacts customer review scores', pad=12)
plt.tight_layout()
plt.savefig('outputs/fig_delay_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. NLP — Negative Review Theme Extraction

TF-IDF on 1-2 star reviews to surface characteristic language,
then mapped to business themes via keyword matching.

In [0]:
display(review_df.head(5))

In [0]:
# ── Translation: Portuguese -> English via Google Translate (deep-translator)
# Resume-safe: JSONL cache skips already-translated reviews on re-run.

!pip install deep-translator  # uncomment if installed

from deep_translator import GoogleTranslator
from pathlib import Path
import json, time

CACHE_FILE = Path('outputs/translation_cache.jsonl')
BATCH_SIZE = 20    # Google Translate handles up to ~5000 chars per call
SLEEP_S    = 0.2   # polite pause between batches

translator = GoogleTranslator(source='pt', target='en')

# ── Load existing cache
cache = {}
if CACHE_FILE.exists():
    with open(CACHE_FILE, encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            cache[rec['original']] = rec['translated']
print(f'Cache loaded: {len(cache):,} existing translations')

# ── Identify texts needing translation
unique_texts = (
    review_df['full_review_text']
    .dropna()
    .pipe(lambda s: s[s.str.len() > 2])
    .unique()
    .tolist()
)
to_translate = [t for t in unique_texts if t not in cache]
print(f'Texts to translate : {len(to_translate):,}')
print(f'Skipping (cached)  : {len(unique_texts) - len(to_translate):,}')

# ── Batch translate
# We join reviews with a separator Google won't mangle,
# then split on the same separator after translation.
SEP = ' ||| '

def translate_batch(texts: list) -> list:
    joined     = SEP.join(texts)
    translated = translator.translate(joined)
    parts      = translated.split(SEP)
    # If separator got mangled, fall back to translating one-by-one
    if len(parts) != len(texts):
        parts = []
        for t in texts:
            try:
                parts.append(translator.translate(t))
            except Exception:
                parts.append(t)   # keep original on failure
    return parts

batches    = [to_translate[i:i + BATCH_SIZE]
              for i in range(0, len(to_translate), BATCH_SIZE)]
total_done = 0

with open(CACHE_FILE, 'a', encoding='utf-8') as cache_f:
    for batch_num, batch in enumerate(batches, 1):
        try:
            translated = translate_batch(batch)
            for orig, trans in zip(batch, translated):
                cache[orig] = trans
                cache_f.write(
                    json.dumps({'original': orig, 'translated': trans},
                               ensure_ascii=False) + '\n'
                )
            total_done += len(batch)
            if batch_num % 10 == 0 or batch_num == len(batches):
                print(f'  Batch {batch_num}/{len(batches)} '
                      f'— {total_done:,} translated so far')
            time.sleep(SLEEP_S)
        except Exception as e:
            print(f'  ERROR batch {batch_num}: {e} — skipping')
            time.sleep(2)

print(f'\nDone. Cache now has {len(cache):,} entries.')

# ── Apply back to dataframe
review_df['full_review_text_en'] = (
    review_df['full_review_text']
    .map(cache)
    .fillna(review_df['full_review_text'])  # fallback: keep original
)

# ── Sanity check
sample = (
    review_df[review_df['full_review_text'].str.len() > 10]
    [['full_review_text', 'full_review_text_en']]
    .dropna()
    .head(3)
)
print('\nSample translations:')
for _, row in sample.iterrows():
    print(f'  PT: {row["full_review_text"][:80]}')
    print(f'  EN: {row["full_review_text_en"][:80]}')
    print()

In [0]:
review_df.to_csv('outputs/translated_reviews.csv', index=False)

In [0]:
neg_df = review_df[
    (review_df['review_score'] <= 2) &
    (review_df['full_review_text_en'].str.len() > 5)
].copy()
print(f'Negative reviews with text: {len(neg_df):,}')

# English stopwords — TF-IDF now runs on translated text
EN_STOPWORDS = [
    'the','a','an','and','or','but','in','on','at','to','for',
    'of','with','it','is','was','i','my','me','this','that',
    'have','had','has','not','no','be','been','are','were',
    'they','he','she','we','you','your','his','her','their',
    'product','order','delivery','received','arrived','item',
    'bought','purchase','store','shop','get','got','came','come',
]

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=200,
    stop_words=EN_STOPWORDS,
    min_df=5,
    sublinear_tf=True,
)
tfidf_matrix  = tfidf.fit_transform(neg_df['full_review_text_en'])
feature_names = tfidf.get_feature_names_out()
mean_scores   = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
term_scores   = pd.Series(mean_scores, index=feature_names).sort_values(ascending=False)

print('Top 30 TF-IDF terms in negative reviews (translated):')
print(term_scores.head(30).to_string())

In [0]:
# English keyword themes — matched against translated text
THEME_KEYWORDS = {
    'Late / non-delivery': [
        'late','delayed','delay','overdue','not arrived',
        'never arrived','still waiting','not delivered',
        'weeks','months waiting','havent received','hasnt arrived',
    ],
    'Damaged / wrong product': [
        'damaged','broken','cracked','dented','defective',
        'wrong item','wrong product','different from','not what i ordered',
        'arrived broken','arrived damaged','came broken',
    ],
    'Poor product quality': [
        'poor quality','bad quality','terrible','awful','horrible',
        'cheap','fragile','fell apart','disappointing','worst',
        'fake','counterfeit','not as described',
    ],
    'Customer service failure': [
        'customer service','support','no response','no reply',
        'ignored','return','refund','exchange','complaint',
        'no one helped','no solution','nobody','unresponsive',
    ],
    'Packaging issues': [
        'packaging','package','box','open','torn','ripped',
        'poorly packed','no protection','unsealed',
    ],
    'Price / value mismatch': [
        'expensive','overpriced','not worth','charged','billing',
        'price','paid too much','shipping cost','hidden fee',
    ],
}

def assign_themes(text):
    text_lower = str(text).lower()
    matched = [t for t, kws in THEME_KEYWORDS.items()
               if any(kw in text_lower for kw in kws)]
    return matched if matched else ['Other']

neg_df['themes'] = neg_df['full_review_text_en'].apply(assign_themes)
themes_exploded  = neg_df.explode('themes')

theme_counts = (
    themes_exploded['themes']
    .value_counts()
    .reset_index()
)
theme_counts.columns = ['theme', 'count']
theme_counts['pct_of_neg_reviews'] = (
    theme_counts['count'] / len(neg_df) * 100
).round(1)

print('Theme frequency in negative reviews:')
print(theme_counts.to_string(index=False))
theme_counts.to_csv('outputs/theme_counts.csv', index=False)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: top 20 TF-IDF terms
ax = axes[0]
top20 = term_scores.head(20)
bar_colors = ['#E8593C' if i < 5 else '#F2A623' if i < 10
              else '#3B8BD4' for i in range(20)]
ax.barh(top20.index[::-1], top20.values[::-1], color=bar_colors[::-1])
ax.set_xlabel('Mean TF-IDF score')
ax.set_title('Top 20 terms in negative reviews (TF-IDF weighted)')

# Right: theme distribution
ax2 = axes[1]
plot_themes = theme_counts[theme_counts['theme'] != 'Other']
ax2.barh(plot_themes['theme'], plot_themes['pct_of_neg_reviews'],
         color=PALETTE[:len(plot_themes)])
ax2.xaxis.set_major_formatter(mticker.PercentFormatter())
ax2.set_xlabel('% of negative reviews mentioning theme')
ax2.set_title('Negative review themes (% of 1-2 star reviews)')

plt.tight_layout()
plt.savefig('outputs/fig_negative_themes.png', dpi=150, bbox_inches='tight')
plt.show()

In [0]:
# Sample verbatims per theme (translated)
print('── Sample verbatims per theme (English) ───────────────────────')
for theme in THEME_KEYWORDS:
    mask    = neg_df['themes'].apply(lambda t: theme in t)
    samples = neg_df.loc[mask, 'full_review_text_en'].dropna().head(3)
    print(f'[{theme}]')
    for s in samples:
        print(f'  - {str(s)[:120]}')
    print()

## 5. Category-Level Review Quality

In [0]:
cat_scores = (
    review_df
    .groupby('product_category_name_english')
    .agg(
        order_count  = ('order_id',    'count'),
        avg_score    = ('review_score','mean'),
        pct_negative = ('score_bin',
                        lambda x: (x == 'Negative (1-2)').mean() * 100),
        avg_delay    = ('delivery_delay_days','mean'),
    )
    .reset_index()
)
cat_scores.columns.name = None

MIN_ORDERS = 100
cat_scores = cat_scores[cat_scores['order_count'] >= MIN_ORDERS].copy()
cat_scores['avg_score']    = cat_scores['avg_score'].round(2)
cat_scores['pct_negative'] = cat_scores['pct_negative'].round(1)
cat_scores['avg_delay']    = cat_scores['avg_delay'].round(1)

print(f'Categories with >= {MIN_ORDERS} orders: {len(cat_scores)}')
print('Worst 10 by avg review score:')
print(
    cat_scores
    .nsmallest(10, 'avg_score')
    [['product_category_name_english','order_count','avg_score','pct_negative','avg_delay']]
    .to_string(index=False)
)
cat_scores.to_csv('outputs/category_scores.csv', index=False)

worst15   = cat_scores.nsmallest(15, 'avg_score')
fig, ax   = plt.subplots(figsize=(9, 6))
bar_colors = ['#E8593C' if v < 3.0 else '#F2A623' if v < 3.5
              else '#3B8BD4' for v in worst15['avg_score']]
bars = ax.barh(worst15['product_category_name_english'],
               worst15['avg_score'], color=bar_colors)
for bar, val in zip(bars, worst15['avg_score']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f} stars', va='center', fontsize=9)
ax.set_xlabel('Avg review score')
ax.set_xlim(0, 5.5)
ax.set_title('Worst-reviewed categories (min 100 orders)', pad=12)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('outputs/fig_category_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Monthly Sentiment Trend

In [0]:
monthly = (
    review_df
    .groupby('year_month')
    .agg(
        order_count  = ('order_id',    'count'),
        avg_score    = ('review_score','mean'),
        pct_negative = ('score_bin',
                        lambda x: (x == 'Negative (1-2)').mean() * 100),
    )
    .reset_index()
)
monthly.columns.name = None
monthly = monthly[monthly['order_count'] >= 50].copy()
monthly['year_month_str'] = monthly['year_month'].astype(str)
monthly.to_csv('outputs/monthly_sentiment.csv', index=False)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.bar(monthly['year_month_str'], monthly['order_count'],
        color='#B4B2A9', alpha=0.5, label='Order volume')
ax2.plot(monthly['year_month_str'], monthly['avg_score'],
         color='#3B8BD4', linewidth=2.5, marker='o', markersize=4,
         label='Avg review score')
ax2.plot(monthly['year_month_str'], monthly['pct_negative'] / 10,
         color='#E8593C', linewidth=2, linestyle='--', marker='x',
         markersize=5, label='% Negative / 10 (scaled)')

tick_step = max(1, len(monthly) // 10)
ax1.set_xticks(range(0, len(monthly), tick_step))
ax1.set_xticklabels(
    monthly['year_month_str'].iloc[::tick_step],
    rotation=30, ha='right', fontsize=9
)
ax1.set_ylabel('Order count', color='#888780')
ax2.set_ylabel('Avg score / scaled % negative', color='#3B8BD4')
ax2.set_ylim(0, 5.5)

l1, lb1 = ax1.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, lb1 + lb2, loc='lower left', fontsize=9)
ax1.set_title('Monthly sentiment trend', pad=12)
plt.tight_layout()
plt.savefig('outputs/fig_monthly_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Output Summary

All files written to `/outputs/`:

| File | Tableau section |
|------|----------------|
| `review_kpis.csv` | Section 1 — Executive KPIs |
| `delay_impact.csv` | Section 2 — Operations |
| `theme_counts.csv` | Section 2 — Operations |
| `category_scores.csv` | Section 3 — Categories |
| `monthly_sentiment.csv` | Section 1 + Section 4 |

**Next steps:**
- Cross-reference `delay_impact.csv` with `category_scores.csv` to identify categories with both worst scores and highest delays